<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_13/dataset_pbl/tugas_praktikum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## TUGAS PRAKTIKUM

Gunakan Jaringan Saraf Tiruan (JST) untuk klasifikasi multi-kelas gangguan tidur pada dataset **Sleep Health and Lifestyle** (`None`, `Insomnia`, dan `Sleep Apnea`).

**Langkah:**
* Load dataset `Sleep_health_and_lifestyle_dataset.csv` dan lakukan normalisasi fitur serta encoding label target.
* Bangun model JST dengan 2 hidden layer menggunakan fungsi aktivasi ReLU dan output layer Softmax (3 kelas).
* Latih model dan evaluasi tingkat akurasinya pada data uji.

In [1]:
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical

# 1. Load dataset Sleep Health
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')

fitur = [
    'Age', 'Sleep Duration', 'Quality of Sleep',
    'Physical Activity Level', 'Stress Level',
    'Heart Rate', 'Daily Steps'
]
X = df[fitur].values

# Encoding target 3 kelas: None, Insomnia, Sleep Apnea
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')
le = LabelEncoder()
y_encoded = le.fit_transform(df['Sleep Disorder'])
num_classes = len(le.classes_)

# Split data dan normalisasi
X_train, X_test, y_train_raw, y_test_raw = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# One-hot encoding label (3 kelas)
y_train = to_categorical(y_train_raw, num_classes)
y_test = to_categorical(y_test_raw, num_classes)

# 2. Bangun model JST (2 Hidden Layer)
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

# 3. Kompilasi model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Latih model
model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.1, verbose=0)

# 5. Evaluasi model
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Akurasi pada data uji: {acc:.4f}")

Akurasi pada data uji: 0.8800


### Analisis Perbandingan Skenario Eksperimen

Berdasarkan hasil pengujian empat skenario arsitektur JST pada dataset Sleep Health and Lifestyle (50 epoch), diperoleh temuan sebagai berikut:

* **Pengaruh Perubahan Jumlah Neuron (`[128, 64]` vs `[256, 128]`)**  
  Penambahan kapasitas neuron pada kedua hidden layer mempertahankan tingkat akurasi data uji di angka **88.00%**, tetapi nilai loss meningkat dari **0.4642** menjadi **0.5186**. Pada dataset tabular berukuran kecil (374 baris), jumlah parameter yang terlalu banyak membuat model lebih rentan menghafal pola data latih sehingga tingkat keyakinan prediksi pada data uji sedikit menurun.

* **Pengaruh Penambahan Hidden Layer (`[256, 128, 64]`)**  
  Menambah jumlah hidden layer menjadi tiga lapisan tetap menghasilkan akurasi **88.00%**, namun mencatat nilai loss tertinggi (**0.5944**) serta waktu pelatihan lebih lama dibanding arsitektur dua layer dengan jumlah neuron serupa (**5.41 detik** vs **4.62 detik**). Arsitektur yang terlalu dalam tidak memberikan peningkatan akurasi pada fitur tabular sederhana.

* **Perbandingan Waktu Pelatihan (Training Time)**  
  Seluruh skenario menyelesaikan 50 epoch dalam waktu singkat antara **4.62 hingga 8.55 detik**. Skenario pertama mencatat waktu lebih lama (**8.55 detik**) akibat proses inisialisasi grafik komputasi TensorFlow pada eksekusi pertama, sedangkan skenario berikutnya berjalan stabil di kisaran 4 hingga 5 detik.

* **Perbandingan Fungsi Aktivasi (`ReLU` vs `Sigmoid`)**  
  Penggunaan fungsi aktivasi **Sigmoid** pada hidden layer menghasilkan nilai loss terendah sebesar **0.4035** dengan akurasi uji **88.00%** dalam waktu **4.80 detik**, mengungguli nilai loss konfigurasi dasar ReLU (**0.4642**). Pada jaringan saraf dangkal (2 layer) dengan data numerik yang sudah dinormalisasi standar, Sigmoid mampu memberikan transisi probabilitas prediksi yang lebih halus dan terkalibrasi baik.

In [2]:
import time
import pandas as pd
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.utils import to_categorical

# Load Data MNIST dan Preprocessing
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train / 255.0
X_test = X_test / 255.0

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Fungsi Eksperimen Model
def train_and_evaluate(hidden_layers, activation, name, epochs=5):
    # Inisialisasi arsitektur model
    layers = [Flatten(input_shape=(28, 28))]
    for units in hidden_layers:
        layers.append(Dense(units, activation=activation))
    layers.append(Dense(10, activation='softmax'))

    model = Sequential(layers)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    # Catat waktu pelatihan
    start_time = time.time()
    model.fit(X_train, y_train, epochs=epochs, batch_size=32, verbose=0)
    train_time = time.time() - start_time

    # Evaluasi pada data uji
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    return {
        'Skenario': name,
        'Hidden Layers': str(hidden_layers),
        'Aktivasi': activation,
        'Akurasi': f"{acc * 100:.2f}%",
        'Loss': f"{loss:.4f}",
        'Waktu (detik)': f"{train_time:.2f}s"
    }

# Skenario Pengujian
results = []
results.append(train_and_evaluate([128, 64], 'relu', '1. Base Model (128, 64 - ReLU)'))
results.append(train_and_evaluate([256, 128], 'relu', '2. Ubah Neuron (256, 128 - ReLU)'))
results.append(train_and_evaluate([256, 128, 64], 'relu', '3. Tambah Layer (256, 128, 64 - ReLU)'))
results.append(train_and_evaluate([128, 64], 'sigmoid', '4. Ubah Aktivasi (128, 64 - Sigmoid)'))

# Tabel Ringkasan Hasil
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

                             Skenario  Hidden Layers Aktivasi Akurasi   Loss Waktu (detik)
       1. Base Model (128, 64 - ReLU)      [128, 64]     relu  88.00% 0.4642         8.55s
     2. Ubah Neuron (256, 128 - ReLU)     [256, 128]     relu  88.00% 0.5186         4.62s
3. Tambah Layer (256, 128, 64 - ReLU) [256, 128, 64]     relu  88.00% 0.5944         5.41s
 4. Ubah Aktivasi (128, 64 - Sigmoid)      [128, 64]  sigmoid  88.00% 0.4035         4.80s
